In [1]:
pip install ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip install librosa

In [3]:
!pip install praat-textgrids librosa soundfile

  Using cached praat_textgrids-1.4.0-py3-none-any.whl.metadata (9.7 kB)
Using cached praat_textgrids-1.4.0-py3-none-any.whl (25 kB)


In [4]:
!pip install textgrid

In [1]:
import os
from textgrid import TextGrid
import librosa
import soundfile as sf
from datetime import datetime
import pandas as pd

In [ ]:
# ===============================================================
# METADATA EMBEDDED CODE FOR SEGMENTATION OF WAV FILES LOCALLY
# ===============================================================

# =====================================================
# PROJECT PATHS
# =====================================================

base_dir = r"Enter the Address of the Parent folder containing all the folders"

wav_folder = os.path.join(base_dir, "WAV")
textgrid_folder = os.path.join(base_dir, "TextGrid")
output_folder = os.path.join(base_dir, "segmented_audio")

processed_csv = os.path.join(base_dir, "processed_recordings.csv")
metadata_csv = os.path.join(base_dir, "segments_metadata.csv")

# =====================================================
# CREATE OUTPUT FOLDERS
# =====================================================

os.makedirs(output_folder, exist_ok=True)

tiers = {
    "ANT": 0,
    "ANI": 1,
    "HUM": 2
}

for label in tiers:
    os.makedirs(os.path.join(output_folder, label), exist_ok=True)

# =====================================================
# LOAD PREVIOUSLY PROCESSED RECORDINGS
# =====================================================

if os.path.exists(processed_csv):

    processed_df = pd.read_csv(processed_csv)

    processed_files = set(processed_df["Recording"])

else:

    processed_df = pd.DataFrame(columns=["Recording", "Processed_Time"])

    processed_files = set()

# =====================================================
# LOAD EXISTING METADATA
# =====================================================

if os.path.exists(metadata_csv):

    metadata_df = pd.read_csv(metadata_csv)

else:

    metadata_df = pd.DataFrame(columns=[
        "Segment",
        "Recording",
        "Label",
        "Start",
        "End",
        "Duration",
        "Sampling_Rate"
    ])

# =====================================================
# FIND WAV FILES
# =====================================================

wav_files = sorted([
    f for f in os.listdir(wav_folder)
    if f.lower().endswith(".wav")
])

print("="*70)
print(f"Found {len(wav_files)} recordings")
print("="*70)

# =====================================================
# PROCESS EACH RECORDING
# =====================================================

for wav_name in wav_files:

    recording = os.path.splitext(wav_name)[0]

    try:

        if recording in processed_files:
            print(f"Skipping {recording} (already processed)")
            continue

        wav_path = os.path.join(wav_folder, wav_name)
        tg_path = os.path.join(textgrid_folder, recording + ".TextGrid")

        if not os.path.exists(tg_path):
            print(f"❌ Missing TextGrid: {recording}.TextGrid")
            continue

        print("\n" + "="*70)
        print(f"Processing {recording}")
        print("="*70)

        audio, sr = librosa.load(wav_path, sr=None)
        tg = TextGrid.fromFile(tg_path)

    # ---------------------------------------------
    # Process every tier
    # ---------------------------------------------

        for label, tier_index in tiers.items():

            tier = tg[tier_index]

            class_folder = os.path.join(output_folder, label)

            existing_files = [
                f for f in os.listdir(class_folder)
                if f.lower().endswith(".wav")
            ]

            clip_number = len(existing_files) + 1

            for interval in tier:

                if interval.mark.strip() != label:
                    continue

                start = interval.minTime
                end = interval.maxTime

                start_sample = int(start * sr)
                end_sample = int(end * sr)

                segment = audio[start_sample:end_sample]

                filename = f"{recording}_{label}_{clip_number:03d}.wav"

                filepath = os.path.join(class_folder, filename)

                sf.write(filepath, segment, sr)

                metadata_df.loc[len(metadata_df)] = [
                    filename,
                    recording,
                    label,
                    start,
                    end,
                    end-start,
                    sr
                ]

                clip_number += 1

        processed_df.loc[len(processed_df)] = [
            recording,
            datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        ]

        processed_files.add(recording)

    except Exception as e:

        print(f"\n❌ Error processing {recording}")
        print(e)

        continue

# =====================================================
# SAVE CSV FILES
# =====================================================

processed_df.to_csv(processed_csv, index=False)

metadata_df.to_csv(metadata_csv, index=False)

print("\n")
print("="*70)
print("SEGMENTATION FINISHED")
print("="*70)

print(f"Processed recordings : {len(processed_df)}")
print(f"Total segments        : {len(metadata_df)}")

print("\nSegments per class")

for label in tiers:

    print(f"{label}: {len(metadata_df[metadata_df['Label'] == label])}")

print("\nMetadata saved.")

print(processed_csv)
print(metadata_csv)

Found 54 recordings

Processing 20251112_202030

Processing 20251112_202042
❌ Missing TextGrid: 20251112_202054.TextGrid

Processing 20251112_202106

Processing 20251112_202118

Processing 20251112_202130

Processing 20251112_202142

Processing 20251112_202154

Processing 20251112_202206

Processing 20251112_202218

Processing 20251112_202230

Processing 20251112_202242

Processing 20251112_202254

Processing 20251112_202306

Processing 20251112_202318

Processing 20251112_202330

Processing 20251112_202342

Processing 20251112_202354

Processing 20251112_202406

Processing 20251112_202418

Processing 20251112_202429

Processing 20251112_202441

Processing 20251112_202452

Processing 20251112_202504

Processing 20251112_202516

Processing 20251112_202528

Processing 20251112_202540

Processing 20251112_202551

Processing 20251112_202603

Processing 20251112_202615

Processing 20251112_202627

Processing 20251112_202639

Processing 20251112_202651

Processing 20251112_202703

Processing